# Lekce 09 - Návrhový vzor metakognice


## Nastavení

Tento notebook demonstruje návrhový vzor metakognice pomocí Microsoft Agent Frameworku.

**Požadavky:**
- Nasazení Azure OpenAI nakonfigurované pomocí proměnných prostředí
- Azure CLI přihlášený (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Co je metakognice?

Metakognice je **myšlení o myšlení**. V kontextu agentů umělé inteligence to znamená vytvářet agenty, kteří dokážou:

- **Sebereflektovat** své vlastní výstupy a proces uvažování
- **Detekovat chyby** a zotavit se elegantně místo tichého selhání
- **Zhodnotit** zda jsou jejich odpovědi úplné a užitečné
- **Přizpůsobit** svou strategii když počáteční přístup nefunguje (např. přepnutí na záložní systém)

Metakognitivní agent nejen odpovídá na otázky — sleduje svůj výkon a pružně se přizpůsobuje.


## Primární a záložní nástroje

Běžným vzorem metakognice je **záložní strategie**. Agent nejdříve zkusí primární nástroj; pokud selže (např. chyba 404), agent rozpozná selhání a transparentně přepne na záložní nástroj.

To odráží systémy z reálného světa, kde primární služby mohou být nedostupné a agenti si musí problém sami diagnostikovat, než zvolí alternativní cestu.

Níže definujeme dva nástroje pro vyhledávání letů:
- **Primární** — pokrývá Paříž, Tokio a Barcelonu
- **Záložní** — pokrývá Berlín, Sydney a New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Sebereflektivní agent s obnovou po chybě

Níže uvedený agent má instrukci nejprve vyzkoušet primární letový systém, rozpoznat selhání a transparentně přejít na záložní systém. Po každé odpovědi krátce provede sebehodnocení, zda plně odpověděl na otázku uživatele.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Vzor sebehodnocení

Další stránkou metakognice je **sebehodnocení**: samostatný agent (nebo tentýž agent při druhém průchodu) posuzuje odpověď z hlediska úplnosti, přesnosti a užitečnosti.

Níže vytváříme agenta `ResponseEvaluator`, který hodnotí odpovědi cestovního agenta ve třech dimenzích.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Shrnutí

V této lekci jste se naučili, jak vytvářet **metakognitivní agenty** pomocí Microsoft Agent Framework:

- **Sebereflexe**: Agenti, kteří sledují své vlastní uvažování a transparentně komunikují, co se stalo.
- **Obnova po chybě s náhradními zdroji**: Vzor primárního + záložního nástroje, kdy agent detekuje selhání (např. chyby 404) a automaticky vyzkouší alternativní zdroj.
- **Sebehodnocení**: Oddělený hodnotící agent, který ohodnotí odpovědi z hlediska úplnosti, přesnosti a užitečnosti.

Tyto vzory činí agenty odolnější, transparentnější a důvěryhodnější — klíčové vlastnosti pro nasazení do produkce.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí služby pro překlad využívající umělou inteligenci [Co-op Translator](https://github.com/Azure/co-op-translator). Ačkoli usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho původním jazyce by měl být považován za rozhodující zdroj. Pro zásadní informace se doporučuje profesionální lidský překlad. Neneseme odpovědnost za jakákoli nedorozumění nebo chybné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
